# StumpVision AI — M4 RF-DETR Cricket Object Detection Training Pipeline

This Google Colab notebook implements the production-grade GPU training workflow for **M4: RF-DETR Cricket Object Detection** in StumpVision AI.

### Key Safeguards & Features:
1. **Isolated Dataset Inputs:** Separate ingestion for public Roboflow Universe datasets and local StumpVision frames.
2. **Zero-Annotation Detection:** Halts with an explicit error if unannotated local frames are supplied.
3. **Explicit Category Normalization:** Normalizes all cricket annotations to the canonical 6 classes (`person`, `stumps`, `pitch`, `crease`, `bat`, `ball`) with an explicit mapping report.
4. **Deep COCO Integrity Validation:** Checks bounding-box bounds, dimensions, duplicate IDs, and missing files before training.
5. **Persistent Google Drive Storage & Auto-Resume:** Saves checkpoints, configs, and metrics directly to Google Drive; auto-resumes from `checkpoint_last.pth` if Colab disconnects.
6. **Conservative T4 GPU Settings:** Sized for Colab free tier (`batch_size=8`, ~7.5 GB VRAM footprint on 15 GB T4).
7. **Pre-Training Dry-Run:** Validates all data and prints split statistics without launching training.

## Cell 01: Environment & GPU Verification

In [ ]:
# Check allocated GPU hardware
!nvidia-smi

import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
else:
    raise SystemError("GPU not detected. Please select: Runtime -> Change runtime type -> T4 GPU")

## Cell 02: Mount Google Drive for Persistent Storage (Safe & Defensive)

- **Privacy & Security:** Isolated strictly to `/MyDrive/stumpvision_m4_rf_detr`.
- **Idempotent:** Does not re-trigger authorization popups if already mounted.
- **Fallback:** If you toggle `USE_GDRIVE = False` or deny authorization, it falls back to local Colab disk without crashing.

In [ ]:
import os
import shutil
from pathlib import Path

# =============================================================================
# Google Drive Persistence Configuration
# =============================================================================
# Set USE_GDRIVE = False if you prefer NOT to grant Google Drive permissions.
# Checkpoints and logs will then be stored on Colab local disk (/content/stumpvision_m4_rf_detr).
USE_GDRIVE = True

GDRIVE_MOUNT_POINT = Path("/content/drive")
PROJECT_SUBDIR_NAME = "stumpvision_m4_rf_detr"

if USE_GDRIVE:
    try:
        # 1. Idempotent check: do not re-prompt if already mounted
        if not (GDRIVE_MOUNT_POINT / "MyDrive").exists():
            print("[INFO] Requesting Google Drive authorization for checkpoint persistence...")
            from google.colab import drive
            drive.mount(str(GDRIVE_MOUNT_POINT), force_remount=False)
        else:
            print("[OK] Google Drive is already mounted.")

        # 2. Scoped directory creation strictly inside our dedicated project folder
        GDRIVE_PROJECT_DIR = GDRIVE_MOUNT_POINT / "MyDrive" / PROJECT_SUBDIR_NAME
        GDRIVE_RUNS_DIR = GDRIVE_PROJECT_DIR / "runs"
        GDRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        GDRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

        # 3. Sandboxed write canary check (verifies write access without touching other files)
        canary_file = GDRIVE_PROJECT_DIR / ".canary_write_test.tmp"
        canary_file.write_text("ok", encoding="utf-8")
        canary_file.unlink()

        # 4. Storage availability audit
        usage = shutil.disk_usage(GDRIVE_PROJECT_DIR)
        free_gb = usage.free / (1024 ** 3)

        print(f"[OK] Google Drive mounted safely.")
        print(f"     Isolated directory: {GDRIVE_PROJECT_DIR}")
        print(f"     Free storage:       {free_gb:.2f} GB available")

    except Exception as exc:
        print(f"[WARNING] Google Drive mount skipped or failed: {exc}")
        print("[INFO] Falling back to local Colab storage safely.")
        USE_GDRIVE = False

if not USE_GDRIVE:
    # Safe fallback to Colab local ephemeral storage
    GDRIVE_PROJECT_DIR = Path("/content") / PROJECT_SUBDIR_NAME
    GDRIVE_RUNS_DIR = GDRIVE_PROJECT_DIR / "runs"
    GDRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    GDRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Active storage: Local Colab disk at {GDRIVE_PROJECT_DIR}")
    print("[NOTE] Checkpoints will be lost if runtime disconnects. Remember to download files manually.")

## Cell 03: Install Dependencies

In [ ]:
# Install RF-DETR with training extras, Roboflow SDK, and supervision
!pip install -q "rfdetr[train]" roboflow supervision matplotlib
print("[OK] RF-DETR and dependencies installed successfully.")

## Cell 04: Training & Dataset Configuration

Configure dataset sources, model capacity, hyperparameters, and resume policy.

In [ ]:
from dataclasses import dataclass, asdict
import json

@dataclass
class M4TrainingConfig:
    # Dataset options
    use_roboflow_dataset: bool = True
    # Default public cricket dataset with verified COCO versions, or any Roboflow Universe slug/URL
    roboflow_project_slug: str = "vignesh-i7zkr/cricket-dataset-pj7rt"
    roboflow_version: int = 1
    
    use_local_dataset: bool = False  # Set True once 54 frames are annotated
    local_dataset_zip: str = "/content/drive/MyDrive/stumpvision_m4_dev_dataset.zip"
    combine_datasets: bool = False
    
    # Training hyperparameters (T4 GPU optimized)
    model_variant: str = "RFDETRMedium"  # RFDETRSmall, RFDETRMedium, or RFDETRBase
    epochs: int = 30
    batch_size: int = 8             # Conservative for T4 15GB VRAM (~7.5GB usage)
    learning_rate: float = 1e-4
    num_classes: int = 6            # 6 canonical classes
    input_size: int = 640
    
    # Persistence and resume
    output_dir: str = str(GDRIVE_RUNS_DIR)
    auto_resume_if_checkpoint_exists: bool = True
    dry_run_only: bool = False      # Set True to validate without training

cfg = M4TrainingConfig()

# Canonical 6 classes
CANONICAL_CATEGORIES = [
    {"id": 1, "name": "person", "supercategory": "cricket"},
    {"id": 2, "name": "stumps", "supercategory": "cricket"},
    {"id": 3, "name": "pitch", "supercategory": "cricket"},
    {"id": 4, "name": "crease", "supercategory": "cricket"},
    {"id": 5, "name": "bat", "supercategory": "cricket"},
    {"id": 6, "name": "ball", "supercategory": "cricket"},
]

# Save effective configuration to Google Drive
with open(GDRIVE_PROJECT_DIR / "effective_training_config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2)
print(f"[OK] Training configuration initialized. Model: {cfg.model_variant}, Batch: {cfg.batch_size}, Epochs: {cfg.epochs}")

## Cell 05: Dataset Acquisition & Provenance Tracking

In [ ]:
import os
import shutil
from pathlib import Path
import getpass

LOCAL_DATA_DIR = Path("/content/stumpvision_dataset")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

dataset_provenance = {}

# Helper to parse Roboflow URLs or slugs
def parse_roboflow_identifier(identifier: str):
    clean = identifier.strip()
    if "universe.roboflow.com/" in clean:
        clean = clean.split("universe.roboflow.com/")[-1]
    clean = clean.strip("/")
    parts = clean.split("/")
    workspace = parts[0]
    project = parts[1] if len(parts) > 1 else ""
    version = None
    if len(parts) >= 3:
        if parts[2] == "dataset" and len(parts) >= 4:
            version = int(parts[3]) if parts[3].isdigit() else parts[3]
        elif parts[2].isdigit():
            version = int(parts[2])
    return workspace, project, version

# 1. Acquire Roboflow Universe Dataset
if cfg.use_roboflow_dataset:
    from roboflow import Roboflow
    print("--- Roboflow Universe Dataset Acquisition ---")
    rf_key = os.environ.get("ROBOFLOW_API_KEY")
    if not rf_key:
        rf_key = getpass.getpass("Enter your free Roboflow API Key: ").strip()
    rf = Roboflow(api_key=rf_key)
    
    workspace_slug, project_slug, parsed_ver = parse_roboflow_identifier(cfg.roboflow_project_slug)
    target_version_num = parsed_ver or cfg.roboflow_version
    
    workspace = rf.workspace(workspace_slug)
    project = workspace.project(project_slug)
    
    # Introspect available versions dynamically
    versions = project.versions()
    if not versions:
        raise RuntimeError(
            f"Project '{workspace_slug}/{project_slug}' has 0 generated dataset versions on Roboflow.\n"
            "Please ensure a version is generated on Roboflow web, or use 'vignesh-i7zkr/cricket-dataset-pj7rt'."
        )
    
    available_names = [getattr(v, 'version', v) for v in versions]
    print(f"[INFO] Found {len(versions)} available version(s): {available_names}")
    
    # Select target version or gracefully fallback to latest
    selected_version = None
    for v in versions:
        v_num_str = str(getattr(v, 'version', '')).split('/')[-1]
        if v_num_str == str(target_version_num) or getattr(v, 'version', None) == target_version_num:
            selected_version = v
            break
            
    if selected_version is None:
        selected_version = versions[-1] # Pick latest generated version
        actual_v = getattr(selected_version, 'version', selected_version)
        print(f"[INFO] Version {target_version_num} not found. Automatically using latest version: {actual_v}")
    else:
        print(f"[INFO] Selected dataset version: {getattr(selected_version, 'version', selected_version)}")
        
    print(f"[INFO] Downloading dataset in COCO format...")
    rf_dataset = selected_version.download("coco")
    
    ROBOFLOW_SOURCE_DIR = Path(rf_dataset.location)
    dataset_provenance["roboflow"] = {
        "source": f"{workspace_slug}/{project_slug}",
        "version": getattr(selected_version, "version", target_version_num),
        "path": str(ROBOFLOW_SOURCE_DIR),
    }
    print(f"[OK] Roboflow dataset acquired at: {ROBOFLOW_SOURCE_DIR}")
else:
    ROBOFLOW_SOURCE_DIR = None

# 2. Acquire Local Dataset (if enabled)
if cfg.use_local_dataset:
    print("\n--- Local StumpVision Dataset Acquisition ---")
    zip_path = Path(cfg.local_dataset_zip)
    if not zip_path.exists():
        raise FileNotFoundError(f"Local dataset zip not found: {zip_path}")
    
    local_extracted = Path("/content/local_extracted")
    !unzip -q "{zip_path}" -d "{local_extracted}"
    LOCAL_SOURCE_DIR = local_extracted
    dataset_provenance["local"] = {
        "source": "StumpVision AI Dev Split",
        "path": str(LOCAL_SOURCE_DIR),
    }
    print(f"[OK] Local dataset unpacked at: {LOCAL_SOURCE_DIR}")
else:
    LOCAL_SOURCE_DIR = None

## Cell 06: Zero-Annotation Detection & Category Normalization

In [ ]:
import json
from copy import deepcopy

# Explicit mapping to canonical 6 classes
EXPLICIT_CATEGORY_MAPPINGS = {
    "batsman": "person",
    "batter": "person",
    "bowler": "person",
    "wicket-keeper": "person",
    "wicketkeeper": "person",
    "wicket keeper": "person",
    "keeper": "person",
    "umpire": "person",
    "leg umpire": "person",
    "main umpire": "person",
    "fielder": "person",
    "fielders": "person",
    "non-striker": "person",
    "player": "person",
    "person": "person",
    "stump": "stumps",
    "stumps": "stumps",
    "wicket": "stumps",
    "wickets": "stumps",
    "bails": "stumps",
    "bail": "stumps",
    "pitch": "pitch",
    "pitch-crease": "pitch",
    "crease": "crease",
    "bat": "bat",
    "cricket-bat": "bat",
    "ball": "ball",
    "cricket-ball": "ball",
}

# Documented unmapped classes to drop safely (e.g. metadata tags or non-target gear)
ALLOWED_UNMAPPED_TO_DROP = {"spectator", "crowd", "helmet", "pad", "gloves", "boundary", "cricket"}

def normalize_and_validate_split(ann_file: Path, split_name: str, check_zero_anns: bool = True):
    with open(ann_file, "r") as f:
        data = json.load(f)
    
    images = data.get("images", [])
    annotations = data.get("annotations", [])
    
    # Rule 3: Zero-annotation detection
    if check_zero_anns and len(annotations) == 0:
        raise ValueError(
            f"[SAFEGUARD HALT] Split '{split_name}' at {ann_file} contains {len(images)} images "
            "but ZERO bounding-box annotations. Unannotated frames cannot be used for object detection training. "
            "Please annotate frames or train on Roboflow data alone."
        )
    
    # Normalization mapping
    src_categories = data.get("categories", [])
    src_id_to_name = {c["id"]: c.get("name", "").strip().lower() for c in src_categories}
    target_name_to_id = {c["name"]: c["id"] for c in CANONICAL_CATEGORIES}
    
    src_id_to_target_id = {}
    unresolved = set()
    mapped_counts = {}
    dropped_counts = {}
    
    for sid, sname in src_id_to_name.items():
        if sname in EXPLICIT_CATEGORY_MAPPINGS:
            tname = EXPLICIT_CATEGORY_MAPPINGS[sname]
            tid = target_name_to_id[tname]
            src_id_to_target_id[sid] = tid
            mapped_counts[sname] = {"target": tname, "target_id": tid, "count": 0}
        elif sname in ALLOWED_UNMAPPED_TO_DROP:
            dropped_counts[sname] = 0
        else:
            unresolved.add(sname)
            
    if unresolved:
        raise ValueError(f"[ERROR] Unresolved categories in '{split_name}': {unresolved}. Explicit mapping required.")
        
    new_annotations = []
    for ann in annotations:
        sid = ann.get("category_id")
        sname = src_id_to_name.get(sid, "unknown")
        if sid in src_id_to_target_id:
            new_ann = deepcopy(ann)
            new_ann["category_id"] = src_id_to_target_id[sid]
            new_annotations.append(new_ann)
            if sname in mapped_counts:
                mapped_counts[sname]["count"] += 1
        elif sname in dropped_counts:
            dropped_counts[sname] += 1
            
    data["categories"] = deepcopy(CANONICAL_CATEGORIES)
    data["annotations"] = new_annotations
    
    with open(ann_file, "w") as f:
        json.dump(data, f, indent=2)
        
    print(f"\n=== Category Normalization Report [{split_name.upper()}] ===")
    for sname, info in mapped_counts.items():
        print(f"  [MAPPED]  {sname:<15} -> {info['target']} (id: {info['target_id']}): {info['count']} boxes")
    for sname, cnt in dropped_counts.items():
        print(f"  [DROPPED] {sname:<15} -> Dropped: {cnt} boxes")
    print(f"  Total Retained: {len(new_annotations)} boxes")

# Execute normalization on active training source
ACTIVE_DATASET_DIR = ROBOFLOW_SOURCE_DIR if cfg.use_roboflow_dataset else LOCAL_SOURCE_DIR
for split in ["train", "valid"]:
    s_ann = ACTIVE_DATASET_DIR / split / "_annotations.coco.json"
    if s_ann.exists():
        normalize_and_validate_split(s_ann, split, check_zero_anns=True)
    else:
        print(f"[WARNING] No _annotations.coco.json in {split}")

## Cell 07: Deep Pre-Training Integrity Validation & Dry-Run

Inspects bounding boxes, dimensions, duplicates, and verifies resume checkpoint status.

In [ ]:
def validate_coco_integrity_deep(split_dir: Path, split_name: str):
    ann_path = split_dir / "_annotations.coco.json"
    with open(ann_path, 'r') as f:
        data = json.load(f)
        
    images = data.get("images", [])
    annotations = data.get("annotations", [])
    categories = data.get("categories", [])
    
    errors = []
    img_map = {}
    img_ids = set()
    img_filenames = set()
    
    for img in images:
        iid = img["id"]
        fname = img["file_name"]
        w = img.get("width", 0)
        h = img.get("height", 0)
        
        if iid in img_ids:
            errors.append(f"Duplicate image ID: {iid}")
        img_ids.add(iid)
        
        if fname in img_filenames:
            errors.append(f"Duplicate image filename: '{fname}'")
        img_filenames.add(fname)
        
        if w <= 0 or h <= 0:
            errors.append(f"Non-positive dimension ({w}x{h}) in image {iid}")
            
        if not (split_dir / fname).exists():
            errors.append(f"Image file missing on disk: '{fname}'")
            
        img_map[iid] = img
        
    ann_ids = set()
    cat_ids = {c["id"] for c in categories}
    cat_counts = {c["name"]: 0 for c in categories}
    name_lookup = {c["id"]: c["name"] for c in categories}
    
    for ann in annotations:
        aid = ann["id"]
        iid = ann["image_id"]
        cid = ann["category_id"]
        bbox = ann.get("bbox", [])
        
        if aid in ann_ids:
            errors.append(f"Duplicate annotation ID: {aid}")
        ann_ids.add(aid)
        
        if iid not in img_ids:
            errors.append(f"Annotation {aid} references nonexistent image ID: {iid}")
            continue
            
        if cid not in cat_ids:
            errors.append(f"Annotation {aid} references unknown category ID: {cid}")
        else:
            cat_counts[name_lookup[cid]] += 1
            
        if len(bbox) != 4:
            errors.append(f"Annotation {aid} has invalid bbox format: {bbox}")
            continue
            
        x, y, w, h = bbox
        if w <= 0 or h <= 0:
            errors.append(f"Annotation {aid} has non-positive bbox width/height: {bbox}")
        if x < 0 or y < 0:
            errors.append(f"Annotation {aid} has negative coordinates: {bbox}")
            
        iw = img_map[iid].get("width", 1920)
        ih = img_map[iid].get("height", 1080)
        if (x + w) > (iw + 1.0) or (y + h) > (ih + 1.0):
            errors.append(f"Annotation {aid} out of bounds ({iw}x{ih}): {bbox}")
            
    print(f"\n--- Pre-Training Dry-Run Validation [{split_name.upper()}] ---")
    print(f"Images: {len(images)}, Total Boxes: {len(annotations)}")
    print("Class Distribution:")
    for cname, cnt in cat_counts.items():
        print(f"  - {cname:<10}: {cnt}")
        
    if errors:
        print(f"[FAIL] {len(errors)} validation errors detected:")
        for e in errors[:5]:
            print(f"  [ERROR] {e}")
        raise ValueError(f"Validation failed for split '{split_name}' with {len(errors)} errors.")
    else:
        print(f"[OK] Split '{split_name}' passed all integrity checks.")

for split in ["train", "valid"]:
    validate_coco_integrity_deep(ACTIVE_DATASET_DIR / split, split)

# Check for existing checkpoint to resume
RESUME_CHECKPOINT = None
if cfg.auto_resume_if_checkpoint_exists:
    potential_ckpts = list(GDRIVE_RUNS_DIR.rglob("*.pth")) + list(GDRIVE_RUNS_DIR.rglob("*.pt"))
    if potential_ckpts:
        potential_ckpts.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        RESUME_CHECKPOINT = str(potential_ckpts[0])
        print(f"\n[RESUME DETECTED] Found existing checkpoint to resume from: {RESUME_CHECKPOINT}")
    else:
        print("\n[INFO] No existing checkpoint found. Training will start from scratch.")

print("\n[READY] Pre-training dry-run complete. System ready for training.")

## Cell 08: Launch RF-DETR Training Loop

Executes fine-tuning with checkpoints saved to Google Drive.

In [ ]:
if cfg.dry_run_only:
    print("[DRY RUN ONLY] dry_run_only is True. Halting before training loop.")
else:
    from rfdetr import RFDETRMedium, RFDETRSmall, RFDETRBase
    
    # Select configured model variant
    model_cls = {
        "RFDETRSmall": RFDETRSmall,
        "RFDETRMedium": RFDETRMedium,
        "RFDETRBase": RFDETRBase,
    }.get(cfg.model_variant, RFDETRMedium)
    
    print(f"Initializing {cfg.model_variant} with num_classes={cfg.num_classes}...")
    model = model_cls(num_classes=cfg.num_classes)
    
    train_kwargs = {
        "dataset_dir": str(ACTIVE_DATASET_DIR),
        "epochs": cfg.epochs,
        "batch_size": cfg.batch_size,
        "output_dir": str(GDRIVE_RUNS_DIR),
    }
    
    # Attach resume path if checkpoint exists
    if RESUME_CHECKPOINT is not None:
        print(f"Resuming training from: {RESUME_CHECKPOINT}")
        train_kwargs["resume"] = RESUME_CHECKPOINT
        
    print(f"Starting training with arguments: {train_kwargs}")
    model.train(**train_kwargs)
    print("[OK] RF-DETR training completed successfully!")

## Cell 09: Validation Inference & Visual Inspection

In [ ]:
import cv2
import matplotlib.pyplot as plt
import supervision as sv

val_images = list((ACTIVE_DATASET_DIR / "valid").glob("*.jpg"))
if val_images and 'model' in locals():
    sample_img_p = str(val_images[0])
    image = cv2.imread(sample_img_p)
    
    # Run prediction with trained model
    detections = model.predict(image, confidence=0.4)
    
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    
    annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"RF-DETR Prediction: {Path(sample_img_p).name}")
    plt.savefig(GDRIVE_PROJECT_DIR / "sample_prediction.png", bbox_inches='tight')
    plt.show()
    print(f"[OK] Sample prediction saved to: {GDRIVE_PROJECT_DIR / 'sample_prediction.png'}")
else:
    print("Skipping visualization: model not in scope or validation images not found.")

## Cell 10: Persistent Checkpoint Summary & Verification

In [ ]:
print("=== Persistent Google Drive Checkpoint Manifest ===")
checkpoints = list(GDRIVE_RUNS_DIR.rglob("*.pth")) + list(GDRIVE_RUNS_DIR.rglob("*.pt"))
if checkpoints:
    for ckpt in checkpoints:
        sz_mb = ckpt.stat().st_size / (1024 * 1024)
        print(f"  - {ckpt.name:<30} ({sz_mb:.1f} MB) -> {ckpt}")
    print("\n[OK] Model weights securely preserved on Google Drive.")
else:
    print("[INFO] No checkpoint files found yet in Google Drive runs directory.")